# Fine-tune mT5-small : Latin → Adlam (Pular)
Runtime → Change runtime type → T4 GPU

In [ ]:
# 1. Install
!pip install -q transformers[torch] datasets accelerate sentencepiece sacrebleu

In [ ]:
# 2. Upload dataset
from google.colab import files
uploaded = files.upload()  # upload train.jsonl, val.jsonl

In [ ]:
# 3. Charger les données
import json

def charger_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

train_data = charger_jsonl('train.jsonl')
val_data   = charger_jsonl('val.jsonl')
print(f'Train: {len(train_data)} | Val: {len(val_data)}')
print('Exemple:', train_data[0])

In [ ]:
# 4. Fine-tune
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments,
                          DataCollatorForSeq2Seq)
from datasets import Dataset
import torch

MODEL = 'google/mt5-small'
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL)

MAX_INPUT  = 128
MAX_TARGET = 128

def tokeniser(batch):
    inputs  = tokenizer(batch['input'],  max_length=MAX_INPUT,  truncation=True, padding=False)
    targets = tokenizer(batch['target'], max_length=MAX_TARGET, truncation=True, padding=False)
    inputs['labels'] = targets['input_ids']
    return inputs

ds_train = Dataset.from_list([{'input': d['input'], 'target': d['target']} for d in train_data])
ds_val   = Dataset.from_list([{'input': d['input'], 'target': d['target']} for d in val_data])
ds_train = ds_train.map(tokeniser, batched=True, remove_columns=['input', 'target'])
ds_val   = ds_val.map(tokeniser,   batched=True, remove_columns=['input', 'target'])

args = Seq2SeqTrainingArguments(
    output_dir='./mt5_adlam',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=5e-4,
    predict_with_generate=True,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    fp16=True,
    logging_steps=50,
    report_to='none',
)

collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    tokenizer=tokenizer,
    data_collator=collator,
)
trainer.train()

In [ ]:
# 5. Test rapide
def transliterer(texte_latin):
    inputs = tokenizer(f'translitere en adlam: {texte_latin}', return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128, num_beams=4)
    return tokenizer.decode(out[0], skip_special_tokens=True)

tests = ['Jam waali', 'Mi yiɗi ɓiɓɓe am', 'Pulaagu woni ndimaagu', 'A jaaraama walaa']
for t in tests:
    print(f'{t}  →  {transliterer(t)}')

In [ ]:
# 6. Télécharger le modèle
import shutil
trainer.save_model('./mt5_adlam_final')
tokenizer.save_pretrained('./mt5_adlam_final')
shutil.make_archive('mt5_adlam_final', 'zip', './mt5_adlam_final')
files.download('mt5_adlam_final.zip')